# PowerBI Data Preparation Notebook

## Overview
This notebook prepares the cleaned Saudi Real Estate market data for PowerBI visualization by creating a **star schema** data model with fact and dimension tables.

### What This Notebook Does
- Loads the master cleaned dataset
- Extracts and exports a **Fact Table** (core metrics)
- Creates **Dimension Tables** (supporting reference data):
  - `dim_city.csv` - City & region information
  - `dim_date.csv` - Time periods (quarters & years)
  - `dim_property_type.csv` - Property classifications
- Documents the **Analysis Scope** and data period coverage
- Exports all tables to `powerbi_data/` folder for PowerBI import

### Output Structure
All generated files are saved in the `powerbi_data/` directory:
```
powerbi_data/
├── fact_real_estate_quarterly.csv    # Main facts table
├── dim_city.csv                       # City dimension
├── dim_date.csv                       # Time dimension
├── dim_property_type.csv              # Property type dimension
└── analysis_scope.csv                 # Analysis scope & notes
```

---

## 1. Setup & Configuration

In [ ]:
# Import required libraries
from pathlib import Path
import pandas as pd

# Configure output directory for PowerBI data files
output_dir = Path("powerbi_data")
output_dir.mkdir(exist_ok=True)

print(f"✓ Output directory ready: {output_dir}")

# Load the cleaned master dataset
master = pd.read_csv("cleaned_dataset/master_real_estate_final.csv")
print(f"✓ Master dataset loaded: {master.shape[0]} rows, {master.shape[1]} columns")

## 2. Fact Table - Core Metrics

**Purpose:** Contains all measurable business metrics (transactions, values, prices, rentals)

**Contains:**
- Time identifiers: `quarter_key`, `year`, `quarter`
- Location: `city`
- Property classification: `property_type`
- Sales metrics: `sales_transactions`, `market_value`, `avg_price_m2`
- Rental metrics: `rental_contracts`, `avg_rent`

This table is the central hub that connects to all dimension tables through foreign keys.

In [ ]:
# Create Fact Table: Main quantitative metrics for real estate market
fact = master[
    [
        "quarter_key",
        "year",
        "quarter",
        "city",
        "property_type",
        "sales_transactions",
        "market_value",
        "avg_price_m2",
        "rental_contracts",
        "avg_rent"
    ]
].copy()

# Export fact table
fact.to_csv(
    output_dir / "fact_real_estate_quarterly.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Fact table exported: {fact.shape[0]} records")
print(f"  Columns: {', '.join(fact.columns.tolist())}")
print(f"  File: powerbi_data/fact_real_estate_quarterly.csv")

## 3. Dimension Tables - Reference Data

Dimension tables provide context and attributes for analyzing the fact table. Each dimension adds a different perspective to the data.

### 3.1 City Dimension

Provides geographic and demographic context for each city
- **Key field:** `city`
- **Attributes:** Region (English), Population (2022 baseline)

In [ ]:
# Create City Dimension Table
# Remove duplicates and sort alphabetically for consistent lookup
dim_city = (
    master[
        [
            "city",
            "region_en",
            "population"
        ]
    ]
    .drop_duplicates(subset=["city"])
    .sort_values("city")
)

# Export city dimension
dim_city.to_csv(
    output_dir / "dim_city.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ City dimension exported: {len(dim_city)} unique cities")
print(f"  Regions: {dim_city['region_en'].nunique()}")
display(dim_city)

### 3.2 Date Dimension

Provides temporal context for time-series analysis
- **Key field:** `quarter_key` (numeric identifier for sorting)
- **Attributes:** Year, Quarter, Quarter Label (e.g., "2022 Q1")
- **Purpose:** Enables proper chronological ordering and time-period filtering in PowerBI

In [ ]:
# Create Date Dimension Table
dim_date = (
    master[
        [
            "quarter_key",
            "year",
            "quarter"
        ]
    ]
    .drop_duplicates()
    .sort_values(["year", "quarter"])
)

# Add user-friendly quarter label (e.g., "2022 Q1")
dim_date["quarter_label"] = (
    dim_date["year"].astype(str)
    + " Q"
    + dim_date["quarter"].astype(str)
)

# Add numeric sort field to ensure proper chronological ordering
dim_date["year_quarter_sort"] = (
    dim_date["year"] * 10 + dim_date["quarter"]
)

# Export date dimension
dim_date.to_csv(
    output_dir / "dim_date.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Date dimension exported: {len(dim_date)} quarters")
print(f"  Period: {dim_date['quarter_label'].min()} to {dim_date['quarter_label'].max()}")
display(dim_date.head(10))

### 3.3 Property Type Dimension

Categorizes properties for segmentation and comparative analysis
- **Key field:** `property_type`
- **Purpose:** Allows filtering and comparing market metrics across different property categories (Apartment, Villa, Land, etc.)

In [ ]:
# Create Property Type Dimension Table
# Simple lookup table sorted alphabetically for consistent reference
dim_property_type = (
    master[["property_type"]]
    .drop_duplicates()
    .sort_values("property_type")
)

# Export property type dimension
dim_property_type.to_csv(
    output_dir / "dim_property_type.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Property type dimension exported: {len(dim_property_type)} categories")
display(dim_property_type)

## 4. Analysis Scope Documentation

This table documents the data availability and analysis periods for different sections of the PowerBI dashboard. It serves as a guide for users to understand which data periods are covered in each analysis area.

In [ ]:
# Create Analysis Scope Reference Table
# This documents which data periods are available for different analysis areas
analysis_scope = pd.DataFrame({
    "analysis_area": [
        "Market Overview",
        "Sales Market",
        "Rental Market",
        "Sales vs Rental Comparison",
        "Price & Growth Analysis",
        "2023 YTD Update",
        "Population"
    ],
    "period_used": [
        "Available data",
        "2019–2022",
        "2019–2024",
        "2019–2022",
        "Controlled by metric availability",
        "2023 Q1–Q3 compared with 2022 Q1–Q3",
        "2022 baseline"
    ],
    "notes": [
        "Broad overview only; sales and rental periods differ.",
        "Unified period for fair sales comparison across cities.",
        "Rental data available through 2024.",
        "Common period between sales and rental data.",
        "Advanced page; requires coverage and outlier checks.",
        "Partial-year comparison only, not full-year comparison.",
        "City-level population from 2022 census."
    ]
})

# Export analysis scope documentation
analysis_scope.to_csv(
    output_dir / "analysis_scope.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Analysis scope documentation exported")
print("\nData Period Coverage Guide:\n")
display(analysis_scope)

## 5. Export Verification

Verify that all required files have been successfully created and exported to the PowerBI data directory.

In [ ]:
# List all exported files in the output directory
print("=" * 60)
print("POWERBI DATA EXPORT SUMMARY")
print("=" * 60)

exported_files = list(output_dir.glob("*.csv"))
print(f"\n✓ Total files exported: {len(exported_files)}\n")

for file in sorted(exported_files):
    file_size = file.stat().st_size / 1024  # Convert to KB
    print(f"  • {file.name:<40} ({file_size:.1f} KB)")

print("\n" + "=" * 60)
print("All data is ready for PowerBI import!")
print("=" * 60)

## 6. Technical Notes & Data Dictionary

### Star Schema Design Pattern
This notebook implements a **star schema** for optimized PowerBI performance:
- **Fact table** at the center containing measurements
- **Dimension tables** radiating outward providing filtering and grouping capabilities
- This design enables fast aggregations and flexible slicing/dicing of data

### Key Field Relationships
```
Fact Table connects to:
├─ dim_city        via 'city' field
├─ dim_date        via 'quarter_key' field
└─ dim_property_type via 'property_type' field
```

### Data Characteristics
- **Time granularity:** Quarterly (Q1-Q4)
- **Geographic scope:** Saudi Arabia cities and regions
- **Data periods:** 
  - Sales data: 2019-2022
  - Rental data: 2019-2024
- **Metrics:** Transaction counts, values, averages (price & rent)

### Import Instructions for PowerBI
1. Open PowerBI Desktop
2. Select **Get Data → CSV**
3. Load files in this order:
   - `dim_date.csv` → Set as **Dimension**
   - `dim_city.csv` → Set as **Dimension**
   - `dim_property_type.csv` → Set as **Dimension**
   - `fact_real_estate_quarterly.csv` → Set as **Fact**
4. Create relationships:
   - `fact.quarter_key` → `dim_date.quarter_key`
   - `fact.city` → `dim_city.city`
   - `fact.property_type` → `dim_property_type.property_type`
5. Hide foreign keys in dimensions (mark as inactive)

### Encoding
All CSV files are encoded in **UTF-8 with BOM** (utf-8-sig) to ensure proper character handling in PowerBI.

### Column Descriptions

#### Fact Table (`fact_real_estate_quarterly.csv`)
| Column | Type | Description |
|--------|------|-------------|
| `quarter_key` | Integer | Numeric ID for quarters (e.g., 201901 for 2019 Q1) - used for proper sorting |
| `year` | Integer | Calendar year (2019-2024) |
| `quarter` | Integer | Quarter number (1-4) |
| `city` | Text | City name (Foreign Key → dim_city) |
| `property_type` | Text | Property category (Foreign Key → dim_property_type) |
| `sales_transactions` | Integer | Number of real estate sales in the quarter |
| `market_value` | Float | Total market value of sales (in millions of currency) |
| `avg_price_m2` | Float | Average price per square meter |
| `rental_contracts` | Integer | Number of rental contracts in the quarter |
| `avg_rent` | Float | Average rental price per unit |

#### City Dimension (`dim_city.csv`)
| Column | Type | Description |
|--------|------|-------------|
| `city` | Text | City name (Primary Key) |
| `region_en` | Text | English region name |
| `population` | Integer | City population (2022 census) |

#### Date Dimension (`dim_date.csv`)
| Column | Type | Description |
|--------|------|-------------|
| `quarter_key` | Integer | Numeric quarter identifier (Primary Key) |
| `year` | Integer | Year value |
| `quarter` | Integer | Quarter number |
| `quarter_label` | Text | User-friendly label (e.g., "2022 Q1") |
| `year_quarter_sort` | Integer | Alternative sort field |

#### Property Type Dimension (`dim_property_type.csv`)
| Column | Type | Description |
|--------|------|-------------|
| `property_type` | Text | Property classification (Primary Key) |

---

## Summary

This notebook prepares cleaned real estate data for PowerBI by:
1. ✅ Creating a fact table with quantitative metrics
2. ✅ Building dimension tables for geographic, temporal, and categorical analysis
3. ✅ Documenting data availability and analysis periods
4. ✅ Exporting all tables in UTF-8 format for PowerBI import

The resulting **star schema** enables efficient analysis, with the fact table at the center and dimensions providing rich context for filtering and aggregation.

### Next Steps
- Run all cells to generate the output files
- Import the CSV files into PowerBI Desktop
- Configure the relationships between fact and dimension tables
- Build visualizations using the clean, well-structured data model